# 10 - Monitoring Reports and Infrastructure Monitoring

This notebook extends the monitoring work completed in `08_monitoring.ipynb` and the
bias monitor in `09_bias_monitoring.ipynb`.

Notebook 08 created CloudWatch prediction logs, custom monitoring metrics, PSI-based drift checks, a CloudWatch dashboard, and CloudWatch alarms. Notebook 09 ran a SageMaker Processing bias monitor and wrote a bias `summary.json` to S3.

This notebook generates formal report artifacts that **embed the live values** pulled back from CloudWatch and S3 (latest metric datapoints, current alarm states, and the latest bias verdict) for:

- Model monitoring
- Data monitoring
- Infrastructure monitoring
- Bias monitoring

The reports are saved locally and uploaded to S3 so they can be referenced in the project README, tracker, and final submission.

## Purpose

This notebook helps satisfy the following monitoring requirements:

1. Implement model monitors on the ML system
2. Implement data monitors on the ML system
3. Implement infrastructure monitors on the ML system
4. Create a monitoring dashboard for the ML endpoint/job on CloudWatch
5. Generate model and data monitoring reports **with real, observed values**

In [1]:
import json
from pathlib import Path
from datetime import datetime, timezone

import boto3
import pandas as pd

print("Imports ready")

Imports ready


In [2]:
with open("project_config.json") as f:
    cfg = json.load(f)

REGION = cfg["REGION"]
SOURCE_BUCKET = cfg["SOURCE_BUCKET"]

s3 = boto3.client("s3", region_name=REGION)
cloudwatch = boto3.client("cloudwatch", region_name=REGION)

print("Region:", REGION)
print("Bucket:", SOURCE_BUCKET)

Region: us-east-1
Bucket: aai540-group1-yelp-data


In [3]:
MONITORING = cfg.get("MONITORING", {})

LOG_GROUP = MONITORING.get("LOG_GROUP", "/yelp-sentiment/predictions")
METRIC_NAMESPACE = MONITORING.get("METRIC_NAMESPACE", "YelpSentiment/Monitoring")
DASHBOARD_NAME = MONITORING.get("DASHBOARD_NAME", "yelp-sentiment-monitoring")
ALARMS = MONITORING.get("ALARMS", [
    "yelp-sentiment-low-confidence",
    "yelp-sentiment-feature-drift"
])

report_timestamp = datetime.now(timezone.utc).strftime("%Y-%m-%d-%H-%M-%S")

print("Log group:", LOG_GROUP)
print("Metric namespace:", METRIC_NAMESPACE)
print("Dashboard:", DASHBOARD_NAME)
print("Alarms:", ALARMS)
print("Report timestamp:", report_timestamp)

Log group: /yelp-sentiment/predictions
Metric namespace: YelpSentiment/Monitoring
Dashboard: yelp-sentiment-monitoring
Alarms: ['yelp-sentiment-low-confidence', 'yelp-sentiment-feature-drift']
Report timestamp: 2026-06-15-04-25-47


In [4]:
from datetime import timedelta

# --- Pull the real, observed values back from CloudWatch and S3 so the
#     reports below contain evidence rather than just descriptions. ---


def latest_metric(metric_name, stat="Average", days=30):
    """Most recent datapoint for a custom metric, or None if none published yet."""
    end = datetime.now(timezone.utc)
    resp = cloudwatch.get_metric_statistics(
        Namespace=METRIC_NAMESPACE,
        MetricName=metric_name,
        StartTime=end - timedelta(days=days),
        EndTime=end,
        Period=3600,
        Statistics=[stat],
    )
    points = sorted(resp.get("Datapoints", []), key=lambda d: d["Timestamp"])
    return round(float(points[-1][stat]), 4) if points else None


def alarm_states(names):
    """Current OK / ALARM / INSUFFICIENT_DATA state for each alarm."""
    if not names:
        return {}
    resp = cloudwatch.describe_alarms(AlarmNames=names)
    return {a["AlarmName"]: a["StateValue"] for a in resp.get("MetricAlarms", [])}


def latest_bias_summary():
    """Read the most recent bias summary.json written by notebook 09, if present."""
    keys = []
    paginator = s3.get_paginator("list_objects_v2")
    for page in paginator.paginate(Bucket=SOURCE_BUCKET, Prefix="bias-monitor/"):
        for obj in page.get("Contents", []):
            if obj["Key"].endswith("summary.json"):
                keys.append((obj["LastModified"], obj["Key"]))
    if not keys:
        return None
    latest_key = sorted(keys)[-1][1]
    body = s3.get_object(Bucket=SOURCE_BUCKET, Key=latest_key)["Body"].read()
    summary = json.loads(body)
    summary["_source_key"] = latest_key
    return summary


def psi_verdict(psi):
    if psi is None:
        return "no data published yet"
    if psi < 0.10:
        return "no significant shift"
    if psi < 0.25:
        return "moderate shift"
    return "significant shift"


live_metrics = {
    "PredictionVolume": latest_metric("PredictionVolume", stat="Sum"),
    "PositivePredictionRate": latest_metric("PositivePredictionRate"),
    "MeanConfidence": latest_metric("MeanConfidence"),
    "MaxFeaturePSI": latest_metric("MaxFeaturePSI"),
}
alarm_state_map = alarm_states(ALARMS)
bias_summary = latest_bias_summary()

print("Live CloudWatch metric values (latest datapoint):")
for name, value in live_metrics.items():
    print(f"  {name:<24} = {value}")

print("\nCurrent alarm states:")
for name, state in (alarm_state_map or {"(none found)": "-"}).items():
    print(f"  {name:<32} = {state}")

print("\nLatest bias summary:")
if bias_summary:
    print(f"  status          = {bias_summary.get('status')}")
    print(f"  violation_count = {bias_summary.get('violation_count')}")
    print(f"  source          = s3://{SOURCE_BUCKET}/{bias_summary.get('_source_key')}")
else:
    print("  (no bias summary.json found under bias-monitor/ — run notebook 09 first)")

Live CloudWatch metric values (latest datapoint):
  PredictionVolume         = 186116.0
  PositivePredictionRate   = 73.538
  MeanConfidence           = 0.918
  MaxFeaturePSI            = 0.0033

Current alarm states:
  yelp-sentiment-feature-drift     = OK
  yelp-sentiment-low-confidence    = OK

Latest bias summary:
  status          = violations_detected
  violation_count = 1
  source          = s3://aai540-group1-yelp-data/bias-monitor/processing-runs/2026-06-15-04-19-29/output/summary.json


## Create Monitoring Report Folder

This section creates a local folder for formal monitoring report artifacts. The reports will later be uploaded to S3 so they can be referenced in the README, tracker, or final project submission.

In [5]:
report_dir = Path("monitoring_reports")
report_dir.mkdir(exist_ok=True)

print("Report folder:", report_dir.resolve())

Report folder: /home/sagemaker-user/monitoring_reports


## Model Monitoring Report

This report documents the model monitoring strategy implemented in Notebook 08.

The monitoring framework tracks:

- Prediction volume
- Positive prediction rate
- Mean prediction confidence
- Prediction logs stored in CloudWatch Logs

These signals help identify model degradation, abnormal prediction behavior, and operational issues.

In [6]:
model_monitoring_report = {
    "report_name": "Model Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "model_monitoring": {
        "prediction_volume": {
            "metric_name": "PredictionVolume",
            "description": "Number of predictions processed.",
            "latest_value": live_metrics["PredictionVolume"],
        },

        "positive_prediction_rate": {
            "metric_name": "PositivePredictionRate",
            "description": "Percentage of positive predictions.",
            "latest_value": live_metrics["PositivePredictionRate"],
        },

        "mean_confidence": {
            "metric_name": "MeanConfidence",
            "description": "Average prediction confidence.",
            "latest_value": live_metrics["MeanConfidence"],
        },

        "prediction_logs": {
            "cloudwatch_log_group": LOG_GROUP
        }
    },

    "cloudwatch_namespace": METRIC_NAMESPACE,

    "dashboard": DASHBOARD_NAME,

    "summary": (
        "Model monitoring is implemented through CloudWatch metrics and "
        "CloudWatch Logs. These metrics provide visibility into prediction "
        "behavior, confidence levels, and operational health. The latest "
        "observed values are recorded above."
    )
}

model_report_file = report_dir / "model_monitoring_report.json"

with open(model_report_file, "w") as f:
    json.dump(model_monitoring_report, f, indent=2)

print("Created:", model_report_file)
print(json.dumps(model_monitoring_report["model_monitoring"], indent=2))

Created: monitoring_reports/model_monitoring_report.json
{
  "prediction_volume": {
    "metric_name": "PredictionVolume",
    "description": "Number of predictions processed.",
    "latest_value": 186116.0
  },
  "positive_prediction_rate": {
    "metric_name": "PositivePredictionRate",
    "description": "Percentage of positive predictions.",
    "latest_value": 73.538
  },
  "mean_confidence": {
    "metric_name": "MeanConfidence",
    "description": "Average prediction confidence.",
    "latest_value": 0.918
  },
  "prediction_logs": {
    "cloudwatch_log_group": "/yelp-sentiment/predictions"
  }
}


## Data Monitoring Report

This report documents the data monitoring strategy implemented in Notebook 08.

Data quality and feature drift are monitored using Population Stability Index (PSI).

The training dataset serves as the baseline distribution, while the production dataset serves as the current distribution.

PSI thresholds:

- PSI < 0.10 → No significant shift
- 0.10 ≤ PSI < 0.25 → Moderate shift
- PSI ≥ 0.25 → Significant shift

In [7]:
data_monitoring_report = {
    "report_name": "Data Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "baseline_dataset": "Training Split",
    "comparison_dataset": "Production Split",

    "monitoring_method": "Population Stability Index (PSI)",

    "features_monitored": cfg["FEATURE_COLS"],

    "drift_thresholds": {
        "psi_less_than_0_10": "No significant shift",
        "psi_0_10_to_0_25": "Moderate shift",
        "psi_greater_than_0_25": "Significant shift"
    },

    "cloudwatch_metric": "MaxFeaturePSI",

    "latest_max_feature_psi": live_metrics["MaxFeaturePSI"],
    "latest_drift_verdict": psi_verdict(live_metrics["MaxFeaturePSI"]),

    "summary": (
        "Data monitoring is implemented using Population Stability Index "
        "(PSI) calculations. Feature distributions in production are "
        "compared against the training baseline. The maximum PSI value "
        "is published to CloudWatch and monitored through alarms. The most "
        "recent observed MaxFeaturePSI and its verdict are recorded above."
    )
}

data_report_file = report_dir / "data_monitoring_report.json"

with open(data_report_file, "w") as f:
    json.dump(data_monitoring_report, f, indent=2)

print("Created:", data_report_file)
print(f"  latest_max_feature_psi = {data_monitoring_report['latest_max_feature_psi']} "
      f"({data_monitoring_report['latest_drift_verdict']})")

Created: monitoring_reports/data_monitoring_report.json
  latest_max_feature_psi = 0.0033 (no significant shift)


## Infrastructure Monitoring Report

This report documents infrastructure monitoring for the SageMaker and CloudWatch components used in the ML system.

Because the project uses SageMaker Batch Transform instead of a persistent real-time endpoint, there is no always-running endpoint to monitor.

Infrastructure monitoring focuses on:

- SageMaker training job status
- SageMaker batch transform job status
- CloudWatch metric publication
- CloudWatch dashboard availability
- CloudWatch alarm configuration

In [8]:
infrastructure_monitoring_report = {
    "report_name": "Infrastructure Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "infrastructure_components": {
        "sagemaker_training": {
            "component": "SageMaker SKLearn Training Job",
            "monitoring_focus": [
                "Training job completion status",
                "Training job failure detection",
                "Model artifact output location"
            ]
        },

        "sagemaker_batch_transform": {
            "component": "SageMaker Batch Transform",
            "monitoring_focus": [
                "Batch transform job completion status",
                "Batch transform output location",
                "Batch inference execution health"
            ]
        },

        "cloudwatch_logs": {
            "component": "CloudWatch Logs",
            "log_group": LOG_GROUP,
            "monitoring_focus": [
                "Prediction event logging",
                "Prediction-level traceability",
                "Recent prediction inspection"
            ]
        },

        "cloudwatch_metrics": {
            "component": "CloudWatch Custom Metrics",
            "namespace": METRIC_NAMESPACE,
            "metrics": [
                "PredictionVolume",
                "PositivePredictionRate",
                "MeanConfidence",
                "MaxFeaturePSI"
            ],
            "latest_values": live_metrics,
        },

        "cloudwatch_dashboard": {
            "dashboard_name": DASHBOARD_NAME,
            "description": "Dashboard used to monitor prediction behavior, model confidence, and feature drift."
        },

        "cloudwatch_alarms": {
            "alarms": ALARMS,
            "current_states": alarm_state_map,
            "description": "Alarms monitor low model confidence and significant feature drift."
        }
    },

    "endpoint_monitoring_note": (
        "This project deploys the model using SageMaker Batch Transform rather than a persistent "
        "real-time endpoint. Therefore, infrastructure monitoring focuses on batch job status, "
        "CloudWatch logs, CloudWatch metrics, dashboards, and alarms instead of endpoint latency "
        "or invocation errors."
    ),

    "summary": (
        "Infrastructure monitoring is implemented through SageMaker job tracking and CloudWatch "
        "observability resources. This provides visibility into the health of training, batch "
        "inference, prediction logging, custom metrics, dashboards, and alarms. The current alarm "
        "states and latest metric values are recorded above."
    )
}

infra_report_file = report_dir / "infrastructure_monitoring_report.json"

with open(infra_report_file, "w") as f:
    json.dump(infrastructure_monitoring_report, f, indent=2)

print("Created:", infra_report_file)
print("  alarm states:", alarm_state_map or "(none found)")

Created: monitoring_reports/infrastructure_monitoring_report.json
  alarm states: {'yelp-sentiment-feature-drift': 'OK', 'yelp-sentiment-low-confidence': 'OK'}


## Bias Monitoring Report

This report captures the latest result from the SageMaker Processing bias monitor
(`09_bias_monitoring.ipynb`). It reads the most recent
`summary.json` written to S3 and records the observed fairness verdict —
demographic parity, disparate impact, and false-negative-rate differences across
the `review_length_group` proxy facet.

In [9]:
bias_monitoring_report = {
    "report_name": "Bias Monitoring Report",
    "created_at_utc": report_timestamp,
    "project": "Yelp Review Sentiment Classification",

    "source": "09_bias_monitoring.ipynb (SageMaker Processing Job)",
    "facet": "review_length_group (proxy facet — dataset has no demographic attributes)",
    "metrics_checked": [
        "disparate_impact_ratio",
        "demographic_parity_difference",
        "false_negative_rate_difference",
    ],

    # Latest observed result pulled from the bias monitor's summary.json in S3
    "latest_run": (
        {
            "status": bias_summary.get("status"),
            "violation_count": bias_summary.get("violation_count"),
            "protected_group": bias_summary.get("protected_group"),
            "comparison_group": bias_summary.get("comparison_group"),
            "current_bias_metrics": bias_summary.get("current_bias_metrics"),
            "source_s3_key": bias_summary.get("_source_key"),
        }
        if bias_summary
        else "No bias summary.json found under bias-monitor/ — run notebook 09 first."
    ),

    "summary": (
        "Bias monitoring is implemented as a SageMaker Processing Job that compares "
        "favorable-outcome and error rates across a facet group between the training "
        "baseline and the production window. The latest observed verdict is recorded "
        "above; demographic parity is flagged when groups differ by more than 10 points."
    )
}

bias_report_file = report_dir / "bias_monitoring_report.json"

with open(bias_report_file, "w") as f:
    json.dump(bias_monitoring_report, f, indent=2)

print("Created:", bias_report_file)
print(json.dumps(bias_monitoring_report["latest_run"], indent=2))

Created: monitoring_reports/bias_monitoring_report.json
{
  "status": "violations_detected",
  "violation_count": 1,
  "protected_group": "short_review",
  "comparison_group": "long_review",
  "current_bias_metrics": {
    "protected_positive_prediction_rate": 0.8044411115277257,
    "comparison_positive_prediction_rate": 0.650753268588368,
    "demographic_parity_difference": 0.15368784293935767,
    "disparate_impact_ratio": 1.2361691448323289,
    "false_negative_rate_difference": -0.02459536505710648
  },
  "source_s3_key": "bias-monitor/processing-runs/2026-06-15-04-19-29/output/summary.json"
}


## Upload Monitoring Reports to S3

Store monitoring reports in S3 so they can be referenced by the project team, README documentation, and final project submission.

In [10]:
report_prefix = f"monitoring-reports/{report_timestamp}"

report_files = [
    model_report_file,
    data_report_file,
    infra_report_file,
    bias_report_file,
]

for report_file in report_files:
    s3_key = f"{report_prefix}/{report_file.name}"

    s3.upload_file(
        str(report_file),
        SOURCE_BUCKET,
        s3_key,
    )

    print(
        f"Uploaded: s3://{SOURCE_BUCKET}/{s3_key}"
    )

Uploaded: s3://aai540-group1-yelp-data/monitoring-reports/2026-06-15-04-25-47/model_monitoring_report.json
Uploaded: s3://aai540-group1-yelp-data/monitoring-reports/2026-06-15-04-25-47/data_monitoring_report.json
Uploaded: s3://aai540-group1-yelp-data/monitoring-reports/2026-06-15-04-25-47/infrastructure_monitoring_report.json


Uploaded: s3://aai540-group1-yelp-data/monitoring-reports/2026-06-15-04-25-47/bias_monitoring_report.json


## Monitoring Report Summary

This notebook extends the monitoring implementation in Notebook 08 and the bias
monitor in Notebook 09. Each report below embeds the **latest observed values**
pulled back from CloudWatch (metric datapoints, alarm states) and S3 (bias summary).

### Monitoring Capabilities

#### Model Monitoring
- Prediction volume, positive prediction rate, mean confidence (latest values embedded)
- CloudWatch prediction logging

#### Data Monitoring
- Population Stability Index (PSI) drift detection (latest MaxFeaturePSI + verdict embedded)
- Production vs training comparison

#### Infrastructure Monitoring
- SageMaker training & batch transform job monitoring
- CloudWatch metrics, alarms (current states embedded), dashboard

#### Bias Monitoring
- SageMaker Processing bias monitor: demographic parity, disparate impact (latest verdict embedded)

### Generated Monitoring Artifacts

| Artifact | Purpose |
|-----------|-----------|
| model_monitoring_report.json | Model monitoring strategy + latest metric values |
| data_monitoring_report.json | Data drift monitoring strategy + latest PSI verdict |
| infrastructure_monitoring_report.json | Infrastructure monitoring + current alarm states |
| bias_monitoring_report.json | Fairness monitoring + latest bias verdict |

### Cloud Resources

- CloudWatch Dashboard, Logs, Custom Metrics, Alarms
- Bias monitor summary in S3
- Monitoring reports stored in S3

This notebook completes the monitoring and reporting layer of the Yelp Sentiment MLOps workflow.

In [11]:
print("=" * 70)
print("YELP SENTIMENT MLOPS MONITORING SUMMARY")
print("=" * 70)

print("\nMonitoring Reports Created:")
print("  • model_monitoring_report.json")
print("  • data_monitoring_report.json")
print("  • infrastructure_monitoring_report.json")
print("  • bias_monitoring_report.json")

print("\nLatest Observed Values:")
for name, value in live_metrics.items():
    print(f"  • {name:<24} = {value}")
print(f"  • Drift verdict           = {psi_verdict(live_metrics['MaxFeaturePSI'])}")
if bias_summary:
    print(f"  • Bias status             = {bias_summary.get('status')} "
          f"({bias_summary.get('violation_count')} violation(s))")

print("\nAlarm States:")
for name, state in (alarm_state_map or {"(none found)": "-"}).items():
    print(f"  • {name:<32} = {state}")

print("\nS3 Report Location:")
print(f"  s3://{SOURCE_BUCKET}/{report_prefix}/")

print("\nStatus: COMPLETE")
print("=" * 70)

YELP SENTIMENT MLOPS MONITORING SUMMARY

Monitoring Reports Created:
  • model_monitoring_report.json
  • data_monitoring_report.json
  • infrastructure_monitoring_report.json
  • bias_monitoring_report.json

Latest Observed Values:
  • PredictionVolume         = 186116.0
  • PositivePredictionRate   = 73.538
  • MeanConfidence           = 0.918
  • MaxFeaturePSI            = 0.0033
  • Drift verdict           = no significant shift
  • Bias status             = violations_detected (1 violation(s))

Alarm States:
  • yelp-sentiment-feature-drift     = OK
  • yelp-sentiment-low-confidence    = OK

S3 Report Location:
  s3://aai540-group1-yelp-data/monitoring-reports/2026-06-15-04-25-47/

Status: COMPLETE
